# 01 — Data loading and quality audit

This notebook loads the synthetic Duka sales data, checks six data-quality dimensions, applies reusable cleaning rules, and saves the cleaned table and cleaning log.

In [ ]:
import pandas as pd
import sys
from pathlib import Path
ROOT = Path.cwd()
if ROOT.name == 'notebooks': ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from src.cleaning import clean_sales

sales_path = ROOT / 'data/raw/sales.csv'
raw = pd.read_csv(sales_path)
raw.head()

## Six-dimension quality audit

We assess completeness, uniqueness, validity, consistency, accuracy of derived totals, and timeliness/date coverage.

In [ ]:
audit = pd.DataFrame({
 'dimension':['Completeness','Uniqueness','Validity','Consistency','Accuracy','Timeliness'],
 'check':['Missing cells','Duplicate transaction IDs','Positive quantity and valid discount','Standardised branch and ID text','Net sales equals gross less discount','Dates within the 2025 analysis period'],
 'result':[
  f"{raw.isna().sum().sum()} missing cells",
  f"{raw.transaction_id.duplicated().sum()} duplicate IDs",
  f"{((raw.quantity<=0)|(raw.discount_rate<0)|(raw.discount_rate>1)).sum()} invalid quantity/discount rows",
  f"{raw.branch.nunique()} branch labels",
  f"{((raw.gross_sales_kes-raw.discount_kes-raw.net_sales_kes).abs()>0.01).sum()} calculation mismatches",
  f"{raw.date.min()} to {raw.date.max()}"
 ]})
audit

In [ ]:
clean, cleaning_log = clean_sales('../data/raw/sales.csv', '../data/processed/sales_clean.csv')
cleaning_log.to_csv('../data/processed/cleaning_log.csv', index=False)
print('Rows before:', len(raw), '| Rows after:', len(clean))
cleaning_log

## Load cleaned data into SQLite

The project database is stored at `data/duka_retail.db`.

In [ ]:
import sqlite3
conn = sqlite3.connect('../data/duka_retail.db')
clean.to_sql('sales_clean', conn, if_exists='replace', index=False)
products = pd.read_csv('../data/raw/products.csv')
customers = pd.read_csv('../data/raw/customers.csv')
products.to_sql('products', conn, if_exists='replace', index=False)
customers.to_sql('customers', conn, if_exists='replace', index=False)
conn.close()
print('SQLite tables created.')